In [0]:
from pyspark.sql.functions import col, lower, trim, when, current_timestamp, coalesce, lit, expr
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
print(" Silver schema ready")

In [0]:
#  clean payments 
payments_clean = (
    spark.table("azure_blob_storage.src_payments")
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("payment_date", 
        expr("""
            coalesce(
                try_to_date(payment_date, 'yyyy/MM/dd'),
                try_to_date(payment_date, 'yyyy/M/d'),
                try_to_date(payment_date, 'yyyy-MM-dd'),
                try_to_date(payment_date, 'MM/dd/yyyy')
            )
        """)
    )
    .withColumn("payment_method", coalesce(lower(trim(col("payment_method"))), lit("none")))
    .withColumn("payment_amount", when(col("payment_amount") < 0, 0).otherwise(col("payment_amount")))
    .withColumn("ingestion_ts", current_timestamp())
    .dropDuplicates(["payment_id"])
)

# Join with invoices
payments = (
    payments_clean.alias("pay")
    .join(
        spark.table("silver.invoices").alias("inv"),
        col("pay.invoice_id") == col("inv.invoice_id"),
        "left"
    )
    .select(
        col("pay.payment_id"),
        col("pay.invoice_id"),
        coalesce(col("inv.customer_id"), lit("none")).alias("customer_id"),
        coalesce(col("inv.customer_name"), lit("none")).alias("customer_name"),
        col("pay.payment_date"),
        col("pay.payment_amount"),
        col("pay.payment_method"),
        coalesce(col("inv.currency"), lit("none")).alias("invoice_currency"),
        col("inv.invoice_date"),
        col("pay.ingestion_ts")
    )
)

payments.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.payments")
print(f" payments: {payments.count()} records")

In [0]:
print("\nSample data (with invoice info):")
display(spark.table("silver.payments").limit(5))